# **New York City Yellow Taxi Data**

## Introduction
This is a part of Upgrad assignment, the 1st task is getting 5% data from huge dataset with 12 files, each files have more than 3 million rows.

Purpose of this notebook: show 2 ways of getting sample data. The 2nd way is the upgraded of 1st way with minor change in execution and much faster speed

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is onlu in 2023.

The data for each month is present in a different parquet file. You will get twelve files for each of the months in 2023.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

## **1** Data Preparation

<font color = red>[5 marks]</font> <br>

### Import Libraries

In [ ]:
# Import warnings

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import the libraries you will be using for analysis

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)

numpy version: 2.0.2
pandas version: 2.2.2
matplotlib version: 3.10.0
seaborn version: 0.13.2


### **1.1** Load the dataset
<font color = red>[5 marks]</font> <br>

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [ ]:
# Try loading one file

df = pd.read_parquet('/content/2023-11.parquet')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3302857 entries, 0 to 3339714
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            floa

How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

---

1st method

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time
from tqdm import tqdm

In [ ]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files
import os
start_time = time.time()

# Select the folder having data files
os.chdir('/content/drive/MyDrive/EDA assignment/trip_records')

# Create a list of all the twelve files to read
file_list = os.listdir()

# initialise an empty dataframe
df = pd.DataFrame()

#1st way to create sampled_data by for loop - original method provided by instructor
# iterate through the list of files and sample one by one:

for file_name in tqdm(file_list, desc ='Processing progress first way'):
    try:
        # file path for the current file
        file_path = os.path.join(os.getcwd(), file_name)

        # Reading the current file
        monthly_file = pd.read_parquet(file_path)
        monthly_file['date'] = monthly_file['tpep_pickup_datetime'].dt.date
        monthly_file['hour'] = monthly_file['tpep_pickup_datetime'].dt.hour

        # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
        # After completing iteration through each date, we will append this data to the final dataframe.
        sampled_data = pd.DataFrame()

        # Loop through dates and then loop through every hour of each date
        for day in monthly_file['date'].unique():
          temp_date_df = monthly_file[monthly_file['date'] == day]

            # Iterate through each hour of the selected date
          for hour in temp_date_df['hour'].unique():
            temp_hour_df = temp_date_df[temp_date_df['hour'] == hour]
            if not temp_hour_df.empty:
              # Sample 5% of the hourly data randomly
              sample = temp_hour_df.sample(frac = 0.05, random_state = 42)

              # add data of this hour to the dataframe
              sampled_data = pd.concat([sampled_data, sample])

        # Concatenate the sampled data of all the dates to a single dataframe
        df = pd.concat([df,sampled_data], ignore_index=True)  # we initialised this empty DF earlier

    except Exception as e:
        print(f"Error reading file {file_name}: {e}")

end_time = time.time()
processed_time = end_time - start_time
print(f"Processed time of 1st way: {processed_time}")

Processing progress first way: 100%|██████████| 12/12 [04:54<00:00, 24.56s/it]

Processed time of 1st way: 294.78582072257996


1st method tooks 294s ~ 5mins

In [ ]:
#2nd way to create sampled_data by group by loop - faster
import os
os.chdir('/content/drive/MyDrive/EDA assignment/trip_records')

start_time = time.time()

file_list = os.listdir()

df = []


for file in tqdm(file_list, desc = 'Processing progress second way'):
  try:
    file_path = os.path.join(os.getcwd(), file)
    monthly_file = pd.read_parquet(file_path)

    # create a new column for group hour and date
    group = [monthly_file['tpep_pickup_datetime'].dt.date, monthly_file['tpep_pickup_datetime'].dt.hour]

    #group and sample instead of 2 for loop
    sampled_data = monthly_file.groupby(group, group_keys=False).apply(lambda x: x.sample(frac=0.05, random_state=42)).reset_index(drop=True)
    df.append(sampled_data)

    del monthly_file

  except Exception as e:
    print(f"Error reading file {file_name}: {e}")
df = pd.concat(df, ignore_index=True)

end_time = time.time()
processed_time = end_time - start_time
print(f"Processed time of 2nd way: {processed_time}")

Processing progress second way: 100%|██████████| 12/12 [00:53<00:00,  4.44s/it]

Processed time of 2nd way: 53.42859673500061


2nd method took 53s -> less than 1 min (5 times faster)

The difference: 1st method uses the FOR loop which is quite slow, especially when data become larger.

2nd method create just 1 more column with combination of 2 important factors for sampling: date & hour, then grouping and sampling within these groups

In [ ]:
# Store the df in csv/parquet
df.to_parquet('/content/drive/MyDrive/EDA assignment/final_sampled_taxi_data_2023.parquet')